### MiddleWare

Middleware provides way to more tightly control what happens inside agent. Middleware is useful for following:

- Tracking agent behavior with logging, analytics and debugging
- Transforming prompts, tool selection, and output formatting
- Adding retries, fallbacks and early termination logic
- Applying rate limits, guardrails and PII detection

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preseerving recent messages while compressing older context. Summarization is useful for following:

- Long-running conversations that exceed context windows
- Multi-turn dialogues with extensive history
- Applications where preserving full conversation context matters

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

#Messagebased summarization
agent = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("messages",10), #This summarization is triggered when 10 messages in context
            keep=("messages",4)  #preserve the last 4 messages as it is in context 
        )
    ]
)

In [11]:
### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

In [12]:
# Alternative test data

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='e5eba374-7233-42a3-9cae-21a96e9088e6'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EmAKXgFpFH0TkF8oakYS4zOjNXtjKuN0aQn7C2dzFnOSnBJ4oxO4y9dsJZySQz1vPRcaHbRBDQobgNS7HETy7a8mjehODmaWPJRC0sBd5nfYLiVGaMlOPwUswDPs0eQA3xc='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a16713-8971-7592-bebd-14d7727fa15c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='e5eba374-7233-42a3-9cae-21a96e9088e6'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EmAKXgFpFH0TkF8

### Token Size

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 Star, 3500/night, spa,pool,gym
    2. City Inn - 4 star, 2000/night, business center
    3. Highway lodge - 3star, 1500/night, free wifi
    """

agents = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger = ("tokens",2000),
            keep = ("tokens",1000)
        ),
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

#Token counter(approx)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4chars - 1 token

In [20]:
# Run test
cities = ["Paris", "London", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens , {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~3230 tokens , 6 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is asking for hotel recommendations across various major global cities. The current session intent is to provide curated lists of top-rated hotels in Dubai, categorized by neighborhood and vibe, following previous requests for Paris, London, and New York.\n\n## SUMMARY\n\n- The user has previously requested and received curated hotel recommendations for **Paris**, **London**, and **New York City**.\n- Each city's recommendations were structured by neighborhood/arrondissement and vibe (e.g., luxury, trendy/boutique, classic charm, budget/value) alongside brief neighborhood guides and follow-up questions to narrow down preferences (dates, budget, travel companions).\n- The user has now asked to find hotels in **Dubai**, which is the current pending action.\n- Rejected Options: None.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Generate a curated list o

### Fraction

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 Star, 3500/night, spa,pool,gym
    2. City Inn - 4 star, 2000/night, business center
    3. Highway lodge - 3star, 1500/night, free wifi
    """

agents = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger = ("fraction",0.005),
            keep = ("tokens",0.002)
        ),
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

#Token counter(approx)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4chars - 1 token

In [22]:
# Run test
cities = ["Paris", "London", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens/128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}) , {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~5077 tokens (3.9664%) , 10 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is requesting curated hotel recommendations across various major global cities, categorized by neighborhood/arrondissement and vibe, along with a quick neighborhood guide and follow-up preference questions. The current session intent is to provide a curated list of top-rated hotels in London.\n\n## SUMMARY\n\n- The user has previously requested and received curated hotel recommendations for **Paris**, **Dubai**, **Singapore**, and a repeated request for **Paris** (with earlier mentions of London and NYC also noted in session history context).\n- For each city, recommendations are structured by neighborhood/location and vibe (e.g., luxury/iconic, boutique/trendy, historic charm, mid-range/value) alongside neighborhood guides and follow-up questions to narrow down preferences (dates, budget, travel companions).\n- The current action is to prov

### Human in the Loop Middleware

Pause agent execution for human approval, editing or rejection of tool calls before they execute. Human-in-the-loop is uesful for following:

- High-stakes operations requiring human approval(e.g database writes, financial transactions)
- Compliance workflows where human oversight is mandatory
- Long-running conversations where human feedback guides the agent

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read email by ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [24]:
agent = create_agent(
    model = "google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [25]:
config = {"configurable":{"thread_id":"test-approve"}}

#Step1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='71c73bf3-fd9a-4c0a-994a-5624eda97b25'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_6088328': 'EmAKXgFpFH0T9MoFBNoJgRBN6SJbN6e1mKPgdLq5PMOD/G/mvOGRGoGXyiT8aWiypS92+V7pk40SSSWVtMpaYCcb8Zxs8lQr/r8xe5z9OD1vMQCLboKAzRFdeigZYOUrDsc='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a17d24-2beb-79a0-8d83-1e34bb1fc3a0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_6088328', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 13

In [29]:
from langgraph.types import Command
# Step2 : Approve
if "__interrupt__" in result:
    print("Paused! Approving")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result['messages'][-1].content}")

Paused! Approving
Result: [{'type': 'text', 'text': 'Email sent successfully to john@test.com.', 'extras': {'signature': 'EmAKXgFpFH0TxvRtwoefmwxg7dV0HeGFlgNFuPCMBlD6l2Ckw4tm/0/cN6QM3X7CBqo8FqCB0ZRpJvn520wcD+ZDhG8YMJG7D/uBVk87hWceqOxyUVtc2nWgeZbq7HINjik='}}]


In [30]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='71c73bf3-fd9a-4c0a-994a-5624eda97b25'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_6088328': 'EmAKXgFpFH0T9MoFBNoJgRBN6SJbN6e1mKPgdLq5PMOD/G/mvOGRGoGXyiT8aWiypS92+V7pk40SSSWVtMpaYCcb8Zxs8lQr/r8xe5z9OD1vMQCLboKAzRFdeigZYOUrDsc='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a17d24-2beb-79a0-8d83-1e34bb1fc3a0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_6088328', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 13

In [31]:
config = {"configurable":{"thread_id":"test-reject"}}

#Step1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [32]:
from langgraph.types import Command
# Step2 : Approve
if "__interrupt__" in result:
    print("Paused! Rejecting")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

print(f"Result: {result['messages'][-1].content}")

Paused! Rejecting
Result: [{'type': 'text', 'text': 'The email send action was rejected by the user, so the email was not sent. Let me know if you would like to try again or need help with something else.', 'extras': {'signature': 'EmAKXgFpFH0TcVIgXjmcNBA6xzgj8ZC5lsvUMzA6KzHjFTbXCNTyQ9YWgVZGNEmwQbIzmrVh2jkPTrfzWTMcRA+afVuwqyAWfH8zvK5CCaj1C6Q3SkJpnoz4qQmLm1KNihM='}}]


In [33]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='60223fbc-f799-49c9-ab4f-7483c7143b51'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_3735237': 'EmAKXgFpFH0T+z3wKqUMTDTB1J6yZfn4s/Z0Xs9YrbkQL7vxrgzHTIiu/ClY10YTk7JnPlJ0ZcZjFaCIp6IP1E8lO6WaboSIrw6rVhh27BX4zvt0calqQKv35icOilvx/pk='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a17d2e-2c7f-7ac0-8189-b1946e80abd3-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_3735237', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 13

In [34]:
config = {"configurable":{"thread_id":"test-edit"}}

#Step1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config
)

In [36]:
from langgraph.types import Command
# Step2 : Approve
if "__interrupt__" in result:
    print("Paused! Editing")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"correct@gmail.com",
                                "subject":"Corrected Subject",
                                "body":"This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

print(f"Result: {result['messages'][-1].content}")

Result: [{'type': 'text', 'text': 'The email has been sent successfully (with the adjustments made by the human reviewer).', 'extras': {'signature': 'EmAKXgFpFH0TMauB6n/FmtJuVXHXbCDxHMb9zw1GdcmCHW5qv5MZDRIc0AaKI53Gd88TpUhmaLCjjZPxGUYyqlJJgoKDndUbtY6iSvpAv4Clci6tYctVGGoXomsNHf6xpd4='}}]


In [37]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='3298568f-7f17-4029-b48a-31a30c6062d9'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_3605212': 'EmAKXgFpFH0T+v8DcyhBbSJyuR2BglMvnIVLRFiZzY6QEeaKtNrYg5LcJcDg3gZm/5E+wWw8vv7g5VtnmYa6Dk/mY97RZYgi7ksdI+RXnElX4BX6fgY+fgYmd8Anu5/cIzk='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a17d2f-fa26-7011-9a77-5d14c58afde6-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'call_3605212', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 13